In [ ]:
import os, time
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR, DATA_DIR

pd.set_option('display.float_format', '{:,.2f}'.format)

DTA_DIR     = RAW_DIR / 'TZNPS5_20_11_STATA'
PARQUET_DIR = DATA_DIR / '01_interim' / 'nps5_parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print('DTA source :', DTA_DIR)
print('Parquet dir:', PARQUET_DIR)
print('DTA files  :', len(list(DTA_DIR.glob('*.dta'))))

---
# Part A — Smart Reading: Column Selection & Dtype Hints

**Two habits to apply:**
1. `columns=[...]` — load only what you need
2. Convert low-cardinality codes (region, district) to `"category"` after loading
   For CSV/Excel, pass `dtype={"region": "category"}` directly to `read_csv`/`read_excel`.
   For Stata, run `.astype("category")` after loading.

## A1. Profile the full main file

In [ ]:
# TODO:
# 1. Load TZNPS5_20.dta
# 2. Print shape and total memory in MB
#    df.memory_usage(deep=True).sum() / 1024**2

## A2. Column-selective load with dtype conversion

Task: *"Compute mean household size and plot count by region and district, for completed interviews."*

Columns needed: `interview__id`, `t0_region`, `t0_district`, `nps4_hhsize`, `nps4_nplots`, `int_result`

After loading, convert `t0_region` and `t0_district` to `"category"` — they are low-cardinality
numeric codes that compress well.

In [ ]:
TASK_COLS = ['interview__id', 't0_region', 't0_district',
             'nps4_hhsize', 'nps4_nplots', 'int_result']

# TODO:
# 1. Load TZNPS5_20.dta with columns=TASK_COLS
# 2. Convert t0_region and t0_district to 'category' with .astype()
#    (For CSV/Excel, pass dtype={...} directly to read_csv/read_excel instead)
# 3. Print shape and memory (MB) before and after the astype conversion
# 4. Print the reduction factor vs the full load from A1 (e.g. "117× smaller")

## A3. Grouped analysis from the slim load

In [ ]:
# TODO (uses the slim DataFrame from A2):
# 1. Filter to int_result == 'COMPLETE'
#    (Stata labeled variables load as string categories — print int_result.unique() if unsure)
# 2. groupby ['t0_region', 't0_district']
# 3. Compute: mean of nps4_hhsize, mean of nps4_nplots, count of interview__id
# 4. Sort by mean_hhsize descending, display top 15

---
# Part B — Vectorised Operations

**The rule:** reach for `.apply()` only when no vectorised form exists.

| Task | Slow | Fast |
|---|---|---|
| Conditional column | `df.apply(lambda r: ...)` | `np.where(...)` / `np.select(...)` |
| Age groups / bins | `df.apply(classify)` | `pd.cut(...)` |
| String cleaning | `df.col.apply(str.strip)` | `df.col.str.strip()` |
| Arithmetic with guard | `df.apply(lambda r: r.a/r.b if r.b else None)` | `df.a / df.b.where(df.b != 0)` |

## B1. Benchmark: `.apply()` vs `pd.cut` for age grouping

Load `Calc_Age` from the roster file and classify each member into:
`"child"` (≤ 14), `"working_age"` (15–64), `"elderly"` (≥ 65).

Measure how long `.apply()` takes vs `pd.cut`, then compute the speedup.

In [ ]:
# Load only Calc_Age from the roster (column-selection habit from Part A)
roster = pd.read_stata(
    DTA_DIR / 't2_roster.dta',
    columns=['interview__id', 'Calc_Age'],
).dropna(subset=['Calc_Age'])

print(f'Roster rows: {len(roster):,}')

# TODO:
# 1. Define age_group(age) → 'child' if age <= 14, 'working_age' if <= 64, else 'elderly'
# 2. Time: roster['age_apply'] = roster['Calc_Age'].apply(age_group)
# 3. Time: roster['age_cut'] = pd.cut(
#        roster['Calc_Age'],
#        bins=[-1, 14, 64, 200],
#        labels=['child', 'working_age', 'elderly']
#    )
# 4. Print both durations and speedup factor
# 5. Compare value_counts() — they should match

## B2. Conditional columns: `np.where` and `np.select`

Using the slim DataFrame from A2:
- `is_complete`: 1 if `int_result == 'COMPLETE'`, else 0  →  use `np.where`
- `hh_size_class`: `"small"` (1–3), `"medium"` (4–6), `"large"` (7+)  →  use `np.select`

In [ ]:
# Uses hh_slim from Part A2

# TODO:
# 1. hh_slim['is_complete'] = np.where(condition, value_if_true, value_if_false)
#
# 2. hh_slim['hh_size_class'] = np.select(
#        conditions=[...],   # list of boolean arrays
#        choicelist=[...],   # matching labels
#        default='unknown'
#    )
#    Bins: small = nps4_hhsize <= 3, medium = <= 6, large = > 6
#
# 3. Print value_counts() for both new columns

## B3. Vectorised arithmetic with a guard

In `previousplots.dta` each row is a plot with:
- `prev_reported_area` — self-reported area (hectares)
- `prev_meas_area`     — GPS-measured area (hectares)

Compute the ratio `prev_reported_area / prev_meas_area` *without* dividing by zero.

Vectorised guard: `df['ratio'] = df['reported'] / df['meas'].where(df['meas'] > 0)`

Then show how many plots have ratio > 2 (self-reported more than double the GPS measure).

In [ ]:
plots = pd.read_stata(
    DTA_DIR / 'previousplots.dta',
    columns=['interview__id', 'prev_reported_area', 'prev_meas_area'],
)
print(f'Plot rows: {len(plots):,}')

# TODO:
# 1. Compute plots['area_ratio'] = prev_reported_area / prev_meas_area.where(prev_meas_area > 0)
# 2. Print: total plots, plots with a valid ratio, plots where ratio > 2
# 3. Show the distribution of area_ratio with .describe()

---
# Part C — Chunked Reading (a backup technique)

> **Key constraint:** accumulate `(sum, count)` per group across chunks,
> then compute `mean = sum / count` *after* the loop. "Average of averages" is wrong
> unless every chunk has exactly the same size.

## C1. Chunked mean household size by region

Compute mean `nps4_hhsize` per `t0_region` using the chunk accumulator pattern.
Do **not** store rows or average per-chunk means.

In [ ]:
CHUNK_SIZE = 200   # small to make the loop visible; in practice use 50_000+

# TODO:
# 1. Open an iterator:
#    pd.read_stata(path, chunksize=CHUNK_SIZE, columns=['t0_region', 'nps4_hhsize'])
# 2. In each chunk:
#    a. drop rows where t0_region or nps4_hhsize is null
#    b. groupby t0_region, accumulate sum and count of nps4_hhsize
# 3. After the loop: mean = accumulated_sum / accumulated_count per region
# 4. Print how many chunks were processed
# 5. Display results sorted by mean descending

## C2. What cannot be done in chunks?

Some statistics require the full dataset before computing anything. Fill in the table.

| Statistic | Chunkable? | Reason |
|---|---|---|
| Count of rows per region | | |
| Global median of `nps4_hhsize` | | |
| Sum of `nps4_nplots` per district | | |
| 90th percentile of `Calc_Age` | | |
| Share of households with `nps4_hhsize > 5` | | |

---
# Part D — File Formats: CSV, Excel, and Parquet

| Capability | CSV | Excel (.xlsx) | Parquet |
|---|---|---|---|
| Open with double-click | ✅ | ✅ | ❌ |
| Preserves types | ❌ | ⚠️ partial | ✅ |
| Compression | ❌ | ⚠️ some | ✅ |
| Multiple sheets/tables | ❌ | ✅ | ❌ |
| Fast for large data | ⚠️ | ❌ | ✅ |
| Read only some columns | ❌ | ⚠️ | ✅ |
| Familiar to NSO staff | ✅ | ✅ | ❌ |

## D1. Convert DTA files to Parquet

The one-time cost: convert Stata files to Parquet once, read them many times.
Parquet compresses each column independently, typically reducing file size 5–20×.

**Important:** Stata categorical columns cannot be stored in Parquet. Cast them to `str` first.

Files to convert:
- `TZNPS5_20.dta`     → `hh_main.parquet`
- `t2_roster.dta`     → `hh_roster.parquet`
- `previousplots.dta` → `plots.parquet`

In [ ]:
FILES_TO_CONVERT = [
    ('TZNPS5_20.dta',     'hh_main.parquet'),
    ('t2_roster.dta',     'hh_roster.parquet'),
    ('previousplots.dta', 'plots.parquet'),
]

# TODO:
# For each (dta_name, pq_name):
#   1. Load with pd.read_stata(path, convert_categoricals=False)
#      convert_categoricals=False keeps numeric codes (e.g. int_result=1) instead of
#      converting them to string labels — required for DuckDB integer comparisons later
#   2. Fix any remaining categoricals:
#      for col in df.select_dtypes('category').columns:
#          df[col] = df[col].astype(str)
#   3. Write: df.to_parquet(PARQUET_DIR / pq_name, index=False)
#   4. Print: DTA size (MB), Parquet size (MB), compression ratio
#      Hint: Path.stat().st_size gives bytes

## D2. Column-selective timing: Parquet vs DTA

Time both approaches for reading the 6 task columns.

In [ ]:
TASK_COLS = ['interview__id', 't0_region', 't0_district',
             'nps4_hhsize', 'nps4_nplots', 'int_result']

# TODO:
# 1. Time pd.read_parquet(PARQUET_DIR / 'hh_main.parquet', columns=TASK_COLS)
# 2. Time pd.read_stata(DTA_DIR / 'TZNPS5_20.dta', columns=TASK_COLS)
# 3. Print both durations and speedup factor
# 4. Confirm both DataFrames have the same shape

## D3. Export a summary to CSV and Excel

Compute a region-level summary from `hh_main.parquet` and export it to both CSV and Excel.
These files will be queried in Part E to demonstrate DuckDB across formats.

In [ ]:
CSV_PATH = PARQUET_DIR / 'region_summary.csv'
XLS_PATH = PARQUET_DIR / 'region_summary.xlsx'

# TODO:
# 1. Load hh_main.parquet with columns=['t0_region','nps4_hhsize','nps4_nplots','int_result']
# 2. Filter to int_result == 1
# 3. groupby t0_region → mean_hhsize, mean_nplots, n_hh
# 4. Export with .to_csv(CSV_PATH, index=False)
# 5. Export with .to_excel(XLS_PATH, index=False)
# 6. Print file sizes for both

## D4. Conversion recipes (reference — no TODO)

Keep these for future projects:

```python
# Excel → Parquet
pd.read_excel('src.xlsx').to_parquet('dst.parquet')

# CSV → Parquet (with explicit types)
pd.read_csv('src.csv', dtype={'region_code': 'str'}).to_parquet('dst.parquet')

# Parquet → Excel (for stakeholders)
pd.read_parquet('data.parquet').to_excel('report.xlsx', index=False)

# Parquet → CSV (for sharing)
pd.read_parquet('data.parquet').to_csv('export.csv', index=False)
```

---
# Part E — DuckDB: SQL Across Formats

| Situation | Tool |
|---|---|
| Aggregations over large files | DuckDB |
| Joining files of different formats | DuckDB |
| Reading only a few columns from a wide Parquet file | DuckDB |
| Statistical modelling | Pandas (after DuckDB preprocessing) |
| Quick in-memory DataFrame manipulation | Pandas |

In [ ]:
try:
    import duckdb
    print(f'duckdb {duckdb.__version__} ready')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb
    print(f'duckdb {duckdb.__version__} installed')

HH_PARQUET     = str(PARQUET_DIR / 'hh_main.parquet')
ROSTER_PARQUET = str(PARQUET_DIR / 'hh_roster.parquet')
PLOTS_PARQUET  = str(PARQUET_DIR / 'plots.parquet')
CSV_PATH_STR   = str(CSV_PATH)
XLS_PATH_STR   = str(XLS_PATH)

## E1. Query Parquet — grouped aggregation

Replicate the grouped mean from A3 using a single SQL query on Parquet.
No Python loop, no chunking — DuckDB handles it internally.

In [ ]:
# TODO:
# Write a SQL query on read_parquet(HH_PARQUET) that:
#   - filters int_result = 1, t0_region IS NOT NULL, nps4_hhsize IS NOT NULL
#   - groups by t0_region
#   - computes AVG(nps4_hhsize) AS mean_hhsize, COUNT(*) AS n_hh
#   - orders by mean_hhsize DESC
# Run with duckdb.sql(...).to_df() and display

## E2. JOIN across Parquet files

Join `hh_main.parquet` (one row per household) to `hh_roster.parquet` (one row per member)
on `interview__id`. Compute:

> Mean age (`Calc_Age`) by region for completed interviews (`int_result = 1`),
> along with household count and total member count.

> **Sanity check:** if the joined table has more rows than the left side, you likely have
> duplicates on the join key. Always verify.

In [ ]:
# TODO:
# Write a SQL query:
#   FROM   read_parquet(HH_PARQUET)     AS hh
#   JOIN   read_parquet(ROSTER_PARQUET) AS r ON hh.interview__id = r.interview__id
#   WHERE  hh.int_result = 1 AND r.Calc_Age IS NOT NULL
#   GROUP  BY hh.t0_region
#   SELECT hh.t0_region,
#          AVG(r.Calc_Age)                  AS mean_age,
#          COUNT(DISTINCT hh.interview__id) AS n_households,
#          COUNT(*)                         AS n_members
#   ORDER  BY mean_age DESC

## E3. Query CSV directly

DuckDB can query a CSV file on disk without loading it into Python first.
Query the region summary CSV created in D3 — no `pd.read_csv()` needed.

In [ ]:
# TODO:
# Use duckdb.sql(f"SELECT * FROM '{CSV_PATH_STR}' ORDER BY mean_hhsize DESC").to_df()
# Display all rows
# Print a note: "DuckDB read the CSV; no pd.read_csv() was needed."

## E4. Join formats: Parquet + Excel

DuckDB can join different file formats in a single query.
Use the Excel file from D3 as a lookup table, join it to the Parquet survey file.

**Approach:** read Excel with pandas → register as a DuckDB view → join in SQL.

*(The DuckDB `excel` extension can also read `.xlsx` natively:
`duckdb.sql("INSTALL excel; LOAD excel;")` then `read_xlsx('file.xlsx')`)*

In [ ]:
# TODO:
# 1. Load XLS_PATH with pd.read_excel() → DataFrame called region_lookup
# 2. Register it: duckdb.register('region_lookup', region_lookup)
# 3. Write a SQL query:
#    FROM   read_parquet(HH_PARQUET) AS hh
#    JOIN   region_lookup            AS rl ON hh.t0_region = rl.t0_region
#    WHERE  hh.int_result = 1
#    GROUP  BY hh.t0_region, rl.mean_hhsize
#    SELECT hh.t0_region,
#           rl.mean_hhsize AS benchmark_hhsize,
#           AVG(hh.nps4_hhsize) AS actual_hhsize,
#           COUNT(*) AS n_hh
#    ORDER  BY hh.t0_region
# The two hhsize columns should be identical (same underlying data) — confirms the join works

## E5. Persist a result as Parquet with `COPY TO`

DuckDB can write query results directly to Parquet, bypassing Python memory entirely.
This avoids allocating a large intermediate DataFrame just to call `.to_parquet()` on it.

```sql
COPY (...query...) TO 'output.parquet' (FORMAT PARQUET);
```

Write the region-age summary from E2 to `PARQUET_DIR / 'region_age_summary.parquet'`,
then read it back with pandas to verify.

In [ ]:
OUT_PATH = str(PARQUET_DIR / 'region_age_summary.parquet')

# TODO:
# 1. duckdb.sql(f"COPY (...your E2 query...) TO '{OUT_PATH}' (FORMAT PARQUET)")
# 2. Read back with pd.read_parquet(OUT_PATH)
# 3. Print shape and display first rows

---
# Summary — When to use each technique

Fill in the table based on what you observed in this notebook.

| Situation | Recommended approach |
|---|---|
| Wide file, need 5 of 969 columns | |
| Repeating string codes (region, district) | |
| Classify ages into groups | |
| Same file queried many times | |
| File larger than RAM, need a grouped mean | |
| JOIN two large files without loading either | |
| Query a CSV without loading it | |
| Share results with a non-Python colleague | |